In [2]:
import pandas as pd
from time import time

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer, OneHotEncoder
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


import matplotlib.pyplot as plt
from feature_engine.outliers import Winsorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import cross_val_score


import numpy as np
import torch
import torch.nn as nn

RT = 42
RANDOM_STATE = 42
torch.manual_seed(RT)
np.random.seed(RT)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(DEVICE)

mps


In [3]:
# 0. load data

X_test = pd.read_csv("/Users/fangsiyu/Desktop/SDU-2026-fall-code/AML/kaggle/X_test.csv")

X_train = pd.read_csv("/Users/fangsiyu/Desktop/SDU-2026-fall-code/AML/kaggle/X_train.csv")
y_train = pd.read_csv("/Users/fangsiyu/Desktop/SDU-2026-fall-code/AML/kaggle/y_train.csv")
# ==========================================================================================

X_test['Gender'] = X_test['Gender'].map({'Female': 0, 'Male': 1})

X_train['Gender'] = X_train['Gender'].map({'Female': 0, 'Male': 1})
y_train['Diabetes'] = y_train['Diabetes'].map({'No': 0, 'Yes': 1})
# ==========================================================================================

X_train = X_train.set_index('ID')
X_test = X_test.set_index('ID')
y_train = y_train.set_index('ID')['Diabetes']

assert X_train.index.equals(y_train.index)




# 1. train_test_split

X_t, X_val, y_t, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=y_train
)
print(f"X_t, y_t, {len(X_t), len(y_t)} \nX_val, y_val {len(X_val), len(y_val)}")
print(f"Length of X_test {len(X_test)}")



# preprocessor
from feature_engine.outliers import Winsorizer

BINARY_COLS = [
    'HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'HeartDiseaseorAttack',
    'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare',
    'NoDocbcCost', 'DiffWalk', 'Gender',
]
NUMERIC_COLS = ['MentHlth', 'PhysHlth', 'Education', 'Income']
ONEHOT_COLS = ['GenHlth', 'Age']

# RISK_COLS = ['HighBP', 'HighChol', 'Stroke', 'HeartDiseaseorAttack', 'DiffWalk']


def make_preprocessor():
    return ColumnTransformer(
        [
            ("bmi", Pipeline([
                ("clip", Winsorizer(capping_method="quantiles", tail="right", fold=0.01)),
                ("scale", StandardScaler()),
            ]), ['BMI']),
            ("num", StandardScaler(), NUMERIC_COLS),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ONEHOT_COLS),
        ],
        remainder="passthrough",
    )


SCALER_OPTIONS = [
    StandardScaler(),
    MinMaxScaler(),
    QuantileTransformer(output_distribution="normal", random_state=RT),
]


def scaler_name(s):
    return "-" if s is None else type(s).__name__

X_t, y_t, (45066, 45066) 
X_val, y_val (7953, 7953)
Length of X_test 17673


In [4]:
import numpy as np

df = pd.concat([X_train, y_train], axis=1)

# 按 21 個欄位分組，每組算人數和有糖尿病的比例
groups = df.groupby(list(X_train.columns))['Diabetes'].agg(['size', 'mean'])

# 只留人數 >= 2 的組
groups = groups[groups['size'] > 1]

# 每組最多猜對的人數，加總後除以總人數
best = (groups['size'] * np.maximum(groups['mean'], 1 - groups['mean'])).sum() / groups['size'].sum()

print(f"有完全相同答案的人數: {groups['size'].sum()} / {len(X_train)}")
print(f"這些人理論上最高的 accuracy: {best:.3f}")

有完全相同答案的人數: 2287 / 53019
這些人理論上最高的 accuracy: 0.883


In [3]:
class MLP(nn.Module):
    def __init__(self, n_in, hidden, n_layers, dropout):
        super().__init__()
        layers = []
        d = n_in
        for _ in range(n_layers):
            layers += [nn.Linear(d, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(dropout)]
            d = hidden
        layers.append(nn.Linear(d, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)   # 輸出 logit，形狀 (batch,)


CFG = dict(
    hidden=256, n_layers=3, dropout=0.3,
    lr=1e-3, weight_decay=1e-4,
    batch_size=512, max_epochs=100, patience=10,
)


CFG_BIG = dict(
    hidden=512, n_layers=6, dropout=0.4,
    lr=1e-4, weight_decay=1e-3,
    batch_size=512, max_epochs=300, patience=20,
)

# ===================================================================================================
class ResBlock(nn.Module):
    def __init__(self, d, dropout):
        super().__init__()
        self.f = nn.Sequential(
            nn.BatchNorm1d(d), 
            nn.Linear(d, d), 
            nn.LeakyReLU(), 
            nn.Dropout(dropout), 
            nn.Linear(d, d),
        )

    def forward(self, x):
        return x + self.f(x)          # 捷徑：輸入直接加到區塊的輸出上


class ResNet(nn.Module):
    def __init__(self, n_in, hidden, n_layers, dropout):
        super().__init__()
        self.inp = nn.Linear(n_in, hidden)
        self.blocks = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.BatchNorm1d(hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x):
        return self.head(self.blocks(self.inp(x))).squeeze(1)


CFG_RES = dict(
    arch=ResNet,
    hidden=2048, n_layers=32, dropout=0.4,
    lr=1e-5, weight_decay=1e-3,
    batch_size=512, max_epochs=10000, patience=20,
)

In [4]:
SEED = 12
CFG_NOW = CFG_BIG
RUN_NAME = f"big_seed{SEED}"

def to_tensor(a):
    return torch.tensor(np.asarray(a), dtype=torch.float32, device=DEVICE)


def fit_mlp(X_tr, y_tr, cfg, seed=RT):
    torch.manual_seed(seed)

    # 從這一折的訓練部分切 10% 出來，只用來決定何時停
    X_a, X_es, y_a, y_es = train_test_split(
        X_tr, y_tr, test_size=0.1, random_state=seed, stratify=y_tr
    )
    prep = make_preprocessor()
    Xa, ya = to_tensor(prep.fit_transform(X_a)), to_tensor(y_a)
    Xes, yes = to_tensor(prep.transform(X_es)), to_tensor(y_es)

    model = cfg.get("arch", MLP)(Xa.shape[1], cfg["hidden"], cfg["n_layers"], cfg["dropout"]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    loss_fn = nn.BCEWithLogitsLoss()

    best_loss, best_state, best_epoch, bad = float("inf"), None, 0, 0
    n = len(Xa)
    for epoch in range(cfg["max_epochs"]):
        model.train()
        perm = torch.randperm(n).to(DEVICE)
        for i in range(0, n, cfg["batch_size"]):
            idx = perm[i:i + cfg["batch_size"]]
            if len(idx) < 2:          # BatchNorm 遇到只有 1 筆的 batch 會報錯
                continue
            opt.zero_grad()
            loss = loss_fn(model(Xa[idx]), ya[idx])
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            es_loss = loss_fn(model(Xes), yes).item()

        if es_loss < best_loss:
            best_loss, best_epoch, bad = es_loss, epoch, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg["patience"]:
                break

    model.load_state_dict(best_state)
    return prep, model, best_epoch


def predict_proba_mlp(prep, model, X):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(to_tensor(prep.transform(X)))).cpu().numpy()

In [5]:
cv = StratifiedKFold(5, shuffle=True, random_state=RT)
oof_mlp = pd.Series(np.nan, index=X_t.index)

t0 = time()
for k, (tr_idx, va_idx) in enumerate(cv.split(X_t, y_t)):
    X_tr, X_va = X_t.iloc[tr_idx], X_t.iloc[va_idx]
    y_tr, y_va = y_t.iloc[tr_idx], y_t.iloc[va_idx]

    prep, model, best_epoch = fit_mlp(X_tr, y_tr, CFG_NOW, seed=SEED)
    p = predict_proba_mlp(prep, model, X_va)
    oof_mlp.iloc[va_idx] = p

    acc = accuracy_score(y_va, (p >= 0.5).astype(int))
    print(f"fold {k}: acc={acc:.4f}  best_epoch={best_epoch}  ({time() - t0:.0f}s)")

print(f"MLP CV accuracy: {accuracy_score(y_t, (oof_mlp >= 0.5).astype(int)):.4f}")

oof_mlp.rename("p").to_csv(f"oof_mlp_{RUN_NAME}.csv")

fold 0: acc=0.7538  best_epoch=40  (18s)
fold 1: acc=0.7510  best_epoch=52  (37s)
fold 2: acc=0.7531  best_epoch=47  (54s)
fold 3: acc=0.7507  best_epoch=29  (66s)
fold 4: acc=0.7619  best_epoch=26  (78s)
MLP CV accuracy: 0.7541


In [6]:
# fold 0: acc=0.7544  best_epoch=6  (5s)
# fold 1: acc=0.7496  best_epoch=8  (8s)
# fold 2: acc=0.7537  best_epoch=2  (10s)
# fold 3: acc=0.7527  best_epoch=10  (13s)
# fold 4: acc=0.7617  best_epoch=1  (14s)
# MLP CV accuracy: 0.7544 


# fold 0: acc=0.7538  best_epoch=58  (24s)
# fold 1: acc=0.7536  best_epoch=43  (41s)
# fold 2: acc=0.7564  best_epoch=32  (55s)
# fold 3: acc=0.7532  best_epoch=34  (69s)
# fold 4: acc=0.7609  best_epoch=43  (85s)
# MLP CV accuracy: 0.7556


# fold 0: acc=0.7514  best_epoch=2  (12s)
# fold 1: acc=0.7498  best_epoch=8  (27s)
# fold 2: acc=0.7558  best_epoch=2  (39s)
# fold 3: acc=0.7538  best_epoch=3  (51s)
# fold 4: acc=0.7607  best_epoch=2  (63s)
# MLP CV accuracy: 0.7543

In [7]:
# oof_mlp.rename("p").to_csv("oof_mlp_res32_seed42.csv")

In [8]:
import glob

files = sorted(glob.glob("oof_mlp_big_seed*.csv"))
oofs = {f: pd.read_csv(f, index_col="ID")["p"] for f in files}

# 第一部分：每個 seed 各自的分數
for f, p in oofs.items():
    acc = accuracy_score(y_t, (p.loc[y_t.index] >= 0.5).astype(int))
    print(f"{f}: {acc:.4f}")

# 第二部分：5 個 seed 的機率平均後的分數
p_avg = pd.concat(oofs.values(), axis=1).mean(axis=1)
acc_avg = accuracy_score(y_t, (p_avg.loc[y_t.index] >= 0.5).astype(int))
print(f"\n平均 {len(oofs)} 個 seed: {acc_avg:.4f}")

oof_mlp_big_seed12.csv: 0.7541
oof_mlp_big_seed22.csv: 0.7540
oof_mlp_big_seed42.csv: 0.7556
oof_mlp_big_seed55.csv: 0.7539
oof_mlp_big_seed67.csv: 0.7515

平均 5 個 seed: 0.7549


In [9]:
p_avg.rename("p").to_csv("oof_mlp_big_avg5.csv")

In [12]:
import glob

files = sorted(glob.glob("oof_mlp_big_seed*.csv"))
oofs = {f: pd.read_csv(f, index_col="ID")["p"] for f in files}

# 第一部分：每個 seed 各自的分數
for f, p in oofs.items():
    acc = accuracy_score(y_t, (p.loc[y_t.index] >= 0.5).astype(int))
    print(f"{f}: {acc:.4f}")

# 第二部分：5 個 seed 的機率平均後的分數
p_avg = pd.concat(oofs.values(), axis=1).mean(axis=1)
acc_avg = accuracy_score(y_t, (p_avg.loc[y_t.index] >= 0.5).astype(int))
print(f"\n平均 {len(oofs)} 個 seed: {acc_avg:.4f}")

oof_mlp_big_seed12.csv: 0.7541
oof_mlp_big_seed22.csv: 0.7540
oof_mlp_big_seed42.csv: 0.7556
oof_mlp_big_seed55.csv: 0.7539
oof_mlp_big_seed67.csv: 0.7515

平均 5 個 seed: 0.7549


In [13]:
p_avg.rename("p").to_csv("oof_mlp_big_avg5.csv")

In [15]:
from sklearn.model_selection import cross_val_predict

from catboost import CatBoostClassifier
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

cat = CatBoostClassifier(
    depth=6, iterations=600, l2_leaf_reg=10, learning_rate=0.03,
    random_seed=RT, verbose=0,
)

cv = StratifiedKFold(5, shuffle=True, random_state=RT)
p_cat = cross_val_predict(cat, X_t, y_t, cv=cv, method="predict_proba")[:, 1]
oof_cat = pd.Series(p_cat, index=X_t.index, name="p")

print(f"CatBoost CV: {accuracy_score(y_t, (oof_cat >= 0.5).astype(int)):.4f}")
oof_cat.to_csv("oof_catboost.csv")

CatBoost CV: 0.7572


In [17]:
SEEDS = [12, 22, 42, 55, 67]
p_test = np.zeros(len(X_test))

for s in SEEDS:
    prep, model, best_epoch = fit_mlp(X_train, y_train, CFG_BIG, seed=s)
    p_test += predict_proba_mlp(prep, model, X_test) / len(SEEDS)
    print(f"seed {s}: best_epoch={best_epoch}")

y_pred_test = (p_test >= 0.5).astype(int)

submission = pd.DataFrame({"ID": X_test.index, "Diabetes": y_pred_test})
submission['Diabetes'] = submission['Diabetes'].map({0: 'No', 1: 'Yes'})
submission.to_csv("submission_dl_mlp_avg5.csv", index=False)

print(submission['Diabetes'].value_counts())

seed 12: best_epoch=21
seed 22: best_epoch=40
seed 42: best_epoch=61
seed 55: best_epoch=57
seed 67: best_epoch=45
Diabetes
Yes    9386
No     8287
Name: count, dtype: int64


In [4]:
import numpy as np

df = pd.concat([X_train, y_train], axis=1)

# 按 21 個欄位分組，每組算人數和有糖尿病的比例
groups = df.groupby(list(X_train.columns))['Diabetes'].agg(['size', 'mean'])

# 只留人數 >= 2 的組
groups = groups[groups['size'] > 1]

# 每組最多猜對的人數，加總後除以總人數
best = (groups['size'] * np.maximum(groups['mean'], 1 - groups['mean'])).sum() / groups['size'].sum()

print(f"有完全相同答案的人數: {groups['size'].sum()} / {len(X_train)}")
print(f"這些人理論上最高的 accuracy: {best:.3f}")

NameError: name 'pd' is not defined